# Donor Retention Predictive

## Problem Framing

**Business question:** Who is at risk of donor lapse?

This notebook is the Phase 5 predictive delivery template for `donor_retention`. It is designed to reuse the shared data prep, modeling, evaluation, and deployment artifacts rather than rebuilding pipeline logic inline.


## Predictive Framing

1. Define the operational decision this score should support.
2. Confirm the label timing and leakage assumptions.
3. Decide whether to use the saved model artifacts as-is or retrain with updated configs.


## Data Sources And Joins

Shared references:

* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-3-predictive-pipelines.md`
* `ml/docs/phase-4-modeling-framework.md`


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml.src.pipelines.registry import build_predictive_dataset, load_predictive_pipeline_config

pipeline_name = "donor_retention"
config = load_predictive_pipeline_config(pipeline_name) if True else {}
dataset = build_predictive_dataset(pipeline_name, save_output=False) if True else pd.DataFrame()
dataset.head()


## Shared Prep Imports

Use the shared modules for:

* dataset building
* feature encoding
* baseline model comparison
* cross-validation
* calibration review


In [ ]:
from ml.src.config.paths import REPORTS_DIR

metrics_path = REPORTS_DIR / "evaluation" / f"{pipeline_name}_metrics.json"
comparison_path = REPORTS_DIR / "evaluation" / "phase4_holdout_comparison.csv"
cv_summary_path = REPORTS_DIR / "evaluation" / "phase4_cv_summary.csv"

metrics = json.loads(metrics_path.read_text(encoding="utf-8")) if metrics_path.exists() else {}
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
cv_summary = pd.read_csv(cv_summary_path) if cv_summary_path.exists() else pd.DataFrame()

metrics


## Modeling

Document:

* the selected target and split logic
* baseline models compared
* threshold choice
* any calibration or tuning changes you make beyond the saved artifacts


In [ ]:
comparison.loc[comparison["pipeline_name"] == pipeline_name]


## Evaluation

Review both holdout and cross-validation before writing conclusions. A strong ROC AUC with weak threshold metrics usually means the ranking signal exists but the decision threshold needs more work.


In [ ]:
cv_summary.loc[cv_summary["pipeline_name"] == pipeline_name]


## Business Interpretation

Write the operational takeaway in plain language:

* What does a high score mean for `donor_retention`?
* Which actions should staff take?
* What risks or fairness concerns should be called out?

## Deployment Notes

Recommended UI patterns from the shared app-integration plan:

* one backend ML service area
* one prediction endpoint per pipeline
* shared widgets such as ranked tables, badges, insight cards, and recommendation panels
